# 02 — LangSmith (SaaS)

Same shared agent (`shared/workflow.py`) — same three scenarios — traced through **LangSmith**.

**Instrumentation approach**: LangSmith is built by the LangChain team, so for any LangChain/LangGraph code it's effectively **zero-instrumentation**: set `LANGSMITH_TRACING=true` plus an API key in env, and every chain/agent run is auto-traced.

**Hosting**: SaaS only (free Developer tier: 1 seat, 5k traces/month).

What to look for in the UI after running this:
1. <https://smith.langchain.com> → project named by `LANGSMITH_PROJECT` in your `.env` (defaults to `observability-comparison`)
2. Three top-level runs named `one_tool` / `two_tools` / `three_tools`
3. Click a run → the **tree view** shows the LangGraph subgraph (`agent` / `tools` / `ToolNode`) nested with token / latency stats per node
4. Each tool span shows input + output JSON, every LLM span shows prompt + response + tokens

## 1. Load environment

LangSmith reads its config purely from environment variables — there's no `Client()` to construct for tracing to work.

In [ ]:
import os, sys, pathlib
from dotenv import load_dotenv

ROOT = pathlib.Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")

assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env"
assert os.environ.get("LANGSMITH_API_KEY"), "Set LANGSMITH_API_KEY in .env"

# These four are the entire LangSmith setup:
os.environ["LANGSMITH_TRACING"] = "true"
os.environ.setdefault("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com")
os.environ.setdefault("LANGSMITH_PROJECT", "observability-comparison")

print("Tracing to project:", os.environ["LANGSMITH_PROJECT"])
print("Endpoint           :", os.environ["LANGSMITH_ENDPOINT"])

## 2. Sanity-check the LangSmith client

In [ ]:
from langsmith import Client

ls_client = Client()
# Listing one project is a cheap auth check
list(ls_client.list_projects(limit=1))
print("LangSmith auth OK")

## 3. Build the shared agent — **prompt pulled from LangSmith Prompt Hub, not code**

The agent's system prompt was pushed to LangSmith's Prompt Hub by `notebooks/00_setup_prompts.ipynb`. This cell fetches it back from **LangSmith** at runtime — the Python `SYSTEM_PROMPT` constant in `shared/workflow.py` is not used here. LangSmith stores prompts as LangChain Runnables under git-style commit hashes; any change in the Prompt Hub UI creates a new commit and `pull_prompt` will get the latest by default.

The printed `fetched N chars` line below is the proof: that content came over the wire from LangSmith, not from disk.

In [ ]:
from shared.workflow import build_agent, SCENARIOS, fetch_system_prompt

# Pull the system prompt from LangSmith Prompt Hub.
# The prompt was pushed by notebooks/00_setup_prompts.ipynb — run that
# once before this notebook (or check the Prompts tab in LangSmith).
print("Pulling system prompt from LangSmith Prompt Hub...")
prompt_text = fetch_system_prompt("langsmith")
print(f"  fetched {len(prompt_text)} chars; first line: {prompt_text.splitlines()[0]!r}")

agent = build_agent(prompt_source="langsmith")
for sc in SCENARIOS:
    print(f"  {sc['id']:<12} expected_tool_calls={sc['expected_tool_calls']}  prompt={sc['prompt']!r}")

## 4. Run the three scenarios

No callbacks, no decorators, no manual span creation — `LANGSMITH_TRACING=true` does the work. We only pass `run_name` / `tags` / `metadata` via `config` so the runs are easy to find in the UI.

In [ ]:
from langchain_core.messages import HumanMessage

# Group all 3 scenarios into one thread so they appear together in
# LangSmith's Threads view. LangSmith reads `thread_id` from metadata.
SESSION_ID = "permission-checks-demo"

results = []
for sc in SCENARIOS:
    config = {
        "run_name": sc["id"],
        "tags": ["observability_comparison", "langsmith", sc["id"]],
        "metadata": {
            "scenario_id": sc["id"],
            "expected_tool_calls": sc["expected_tool_calls"],
            # Thread grouping (LangSmith reads `thread_id`)
            "thread_id": SESSION_ID,
            "session_id": SESSION_ID,
            "langfuse_session_id": SESSION_ID,
        },
    }
    out = agent.invoke({"messages": [HumanMessage(content=sc["prompt"])]}, config=config)
    actual = sum(len(getattr(m, "tool_calls", []) or []) for m in out["messages"])
    final = out["messages"][-1].content
    results.append({**sc, "final": final, "actual_tool_calls": actual})
    print(f"\n--- {sc['id']} ---")
    print(f"prompt   : {sc['prompt']}")
    print(f"tools    : expected={sc['expected_tool_calls']}  actual={actual}")
    print(f"final    : {final}")

print(f"\nDone. Runs are in thread '{SESSION_ID}' — open https://smith.langchain.com -> {os.environ['LANGSMITH_PROJECT']} -> Threads.")

## 5. Attach feedback (LangSmith-specific)

LangSmith calls scores **feedback**. The pattern is: capture the run id during execution (via `trace()` context manager or by querying afterwards), then `client.create_feedback(run_id, key, score, ...)`. Below we replay with `tracing_v2_enabled` so we can grab the run id.

In [ ]:
from langchain_core.tracers.context import tracing_v2_enabled

for sc in SCENARIOS:
    with tracing_v2_enabled(
        project_name=os.environ["LANGSMITH_PROJECT"],
    ) as cb:
        out = agent.invoke(
            {"messages": [HumanMessage(content=sc["prompt"])]},
            config={
                "run_name": f"{sc['id']}__scored",
                "tags": ["observability_comparison", "langsmith", "scored"],
                "metadata": {
                    "thread_id": SESSION_ID,
                    "session_id": SESSION_ID,
                    "langfuse_session_id": SESSION_ID,
                },
            },
        )
        run_id = cb.latest_run.id  # id of the top-level run we just produced

    actual = sum(len(getattr(m, "tool_calls", []) or []) for m in out["messages"])
    ls_client.create_feedback(
        run_id=run_id,
        key="tool_call_count_matches",
        score=1.0 if actual == sc["expected_tool_calls"] else 0.0,
        comment=f"expected={sc['expected_tool_calls']} actual={actual}",
    )
    print(f"{sc['id']}: feedback attached to run {run_id}")

print("\nDone. Inspect runs at https://smith.langchain.com")

## Takeaways for the comparison

- **Setup cost**: 3 env vars — that's the entire setup if you're already on LangChain/LangGraph.
- **What you get out of the box**: the most polished LangGraph rendering of the three (it's the same team), with token/cost/latency rollups per node, automatic streaming support, and a `Playground` tab to replay any LLM span.
- **Where it leans in**: prompt hub, evals (LLM-as-judge + heuristic), datasets, prompt versioning, online monitoring.
- **Friction**: SaaS only; vendor-tied to the LangChain ecosystem (works fine outside it via `@traceable`, but less polished than for LangGraph).